# WiFi CSI Indoor Positioning System
## Signal Simulation | ML-Based Positioning | Movement Detection

Simulates WiFi Channel State Information (CSI) for indoor localization without GPS.
Ready to connect to real SGPCard Mini hardware when available.

---

In [ ]:
# CELL 1: Setup
import subprocess, sys
for p in ['numpy','pandas','scikit-learn','matplotlib','plotly','seaborn','scipy']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from scipy import signal as sig
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
OUTPUT_DIR = Path('outputs'); OUTPUT_DIR.mkdir(exist_ok=True)
print('CSI Positioning System Ready!')

## Section 1: CSI Signal Simulation
Simulate 4 WiFi access points measuring Channel State Information.

In [ ]:
# CELL 2: Simulate CSI Data from 4 Access Points
def simulate_csi_signals(n_samples=2000, room_size=(10, 8), n_aps=4):
    """
    Simulate WiFi CSI signals for indoor positioning.
    
    CSI amplitude follows log-distance path loss model:
    RSSI(d) = RSSI(d0) - 10*n*log10(d/d0) + noise
    
    Args:
        n_samples: Number of position measurements
        room_size: Room dimensions in meters (width, height)
        n_aps: Number of access points
    Returns:
        DataFrame with CSI features and true positions
    """
    # Access point locations (corners + center variations)
    ap_positions = np.array([
        [0.5, 0.5], [room_size[0]-0.5, 0.5],
        [0.5, room_size[1]-0.5], [room_size[0]-0.5, room_size[1]-0.5]
    ])[:n_aps]
    
    # Random positions in room
    true_x = np.random.uniform(1, room_size[0]-1, n_samples)
    true_y = np.random.uniform(1, room_size[1]-1, n_samples)
    
    data = {'true_x': true_x, 'true_y': true_y}
    
    for ap_idx in range(n_aps):
        ap = ap_positions[ap_idx]
        distances = np.sqrt((true_x - ap[0])**2 + (true_y - ap[1])**2)
        
        # Path loss model parameters
        rssi_ref = -30  # RSSI at 1m reference distance
        path_loss_exp = 2.5 + np.random.uniform(-0.3, 0.3)  # Path loss exponent
        
        # CSI amplitude (multiple subcarriers)
        for sc in range(8):  # 8 subcarriers per AP
            freq_factor = 1 + 0.05 * sc
            amplitude = rssi_ref - 10 * path_loss_exp * np.log10(distances * freq_factor + 0.1)
            amplitude += np.random.normal(0, 2, n_samples)  # Noise
            # CSI phase (affected by distance and multipath)
            phase = (2 * np.pi * distances * freq_factor / 0.125) % (2 * np.pi)
            phase += np.random.normal(0, 0.3, n_samples)
            
            data[f'ap{ap_idx}_sc{sc}_amp'] = amplitude
            data[f'ap{ap_idx}_sc{sc}_phase'] = phase
    
    return pd.DataFrame(data), ap_positions

df_csi, ap_pos = simulate_csi_signals(n_samples=3000)
print(f"Dataset shape: {df_csi.shape}")
print(f"Features: {len([c for c in df_csi.columns if 'ap' in c])} CSI features")

# Visualize AP layout
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(df_csi['true_x'], df_csi['true_y'], c=df_csi['ap0_sc0_amp'], cmap='viridis', s=5, alpha=0.3)
ax.scatter(ap_pos[:, 0], ap_pos[:, 1], c='red', s=200, marker='^', zorder=5, label='Access Points')
for i, p in enumerate(ap_pos): ax.annotate(f'AP{i}', p, fontsize=12, fontweight='bold', color='red')
ax.set_title('Room Layout with CSI Signal Strength'); ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.legend(); plt.colorbar(ax.collections[0], label='CSI Amplitude (dBm)')
plt.savefig(OUTPUT_DIR / 'room_layout.png', dpi=150); plt.show()

In [ ]:
# CELL 3: ML-Based Position Estimation
feature_cols = [c for c in df_csi.columns if 'ap' in c]
X = df_csi[feature_cols].values
y_x = df_csi['true_x'].values
y_y = df_csi['true_y'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, yx_train, yx_test, yy_train, yy_test = train_test_split(
    X_scaled, y_x, y_y, test_size=0.2, random_state=42)

# Train position regressors
rf_x = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf_y = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf_x.fit(X_train, yx_train); rf_y.fit(X_train, yy_train)

pred_x = rf_x.predict(X_test); pred_y = rf_y.predict(X_test)
errors = np.sqrt((pred_x - yx_test)**2 + (pred_y - yy_test)**2)

print(f"Mean positioning error: {errors.mean():.2f}m")
print(f"Median positioning error: {np.median(errors):.2f}m")
print(f"90th percentile error: {np.percentile(errors, 90):.2f}m")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(yx_test, yy_test, c='blue', s=10, alpha=0.3, label='True')
axes[0].scatter(pred_x, pred_y, c='red', s=10, alpha=0.3, label='Predicted')
axes[0].set_title('Position Estimation'); axes[0].legend()
axes[1].hist(errors, bins=30, color='steelblue', edgecolor='white')
axes[1].axvline(errors.mean(), color='red', linestyle='--', label=f'Mean={errors.mean():.2f}m')
axes[1].set_title('Positioning Error Distribution'); axes[1].legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'positioning_results.png', dpi=150); plt.show()

## Section 2: Activity Recognition from CSI Patterns
Detect standing, sitting, walking from temporal CSI signal variations.

In [ ]:
# CELL 4: Activity Recognition
def generate_activity_csi(n_per_class=500):
    """Simulate CSI patterns for different human activities."""
    activities = {'standing': 0, 'sitting': 1, 'walking': 2, 'waving': 3}
    data = []
    for activity, label in activities.items():
        for _ in range(n_per_class):
            t = np.linspace(0, 2, 100)  # 2 seconds of data
            if activity == 'standing':    signal = np.sin(2*np.pi*0.1*t) * 0.5 + np.random.normal(0, 0.2, 100)
            elif activity == 'sitting':   signal = np.sin(2*np.pi*0.05*t) * 0.3 + np.random.normal(0, 0.15, 100) - 0.5
            elif activity == 'walking':   signal = np.sin(2*np.pi*2*t) * 2 + np.random.normal(0, 0.5, 100)
            elif activity == 'waving':    signal = np.sin(2*np.pi*4*t) * 1.5 + np.sin(2*np.pi*8*t) * 0.5 + np.random.normal(0, 0.3, 100)
            
            # Extract features
            features = {
                'mean': np.mean(signal), 'std': np.std(signal),
                'max': np.max(signal), 'min': np.min(signal),
                'energy': np.sum(signal**2), 'zcr': np.sum(np.diff(np.sign(signal)) != 0),
                'peak_freq': np.abs(np.fft.rfft(signal)).argmax(),
                'spectral_entropy': -np.sum(np.abs(np.fft.rfft(signal))**2 * np.log(np.abs(np.fft.rfft(signal))**2 + 1e-10)),
                'label': label
            }
            data.append(features)
    return pd.DataFrame(data)

df_act = generate_activity_csi()
X_act = df_act.drop('label', axis=1).values
y_act = df_act['label'].values

X_tr, X_te, y_tr, y_te = train_test_split(X_act, y_act, test_size=0.2, random_state=42, stratify=y_act)
clf = GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42)
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

print("Activity Recognition Results:")
print(classification_report(y_te, y_pred, target_names=['Standing','Sitting','Walking','Waving']))
print(f"Accuracy: {accuracy_score(y_te, y_pred):.1%}")
print("\nP2 WiFi CSI Indoor Positioning - COMPLETE!")